In [ ]:
# Install required libraries
!pip install -q --upgrade openai numpy scikit-learn matplotlib


## Tutorial: Generating Embeddings via OpenRouter
In this tutorial you will:
- Configure OpenRouter
- Generate embeddings for short texts
- Normalize and compare vectors
- Visualize clusters in 2D


### 1) Configure OpenRouter API Key
Enter your OpenRouter API key when prompted. Do not hard-code it in the notebook.


In [ ]:
from getpass import getpass
from openai import OpenAI

OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

print("OpenRouter API key loaded:", bool(OPENROUTER_API_KEY))


### 2) Sample texts
We'll embed a tiny set for quick iteration.


In [ ]:
texts = [
    "RAG uses retrieval to ground generation.",
    "Pinecone is a vector database.",
    "Weaviate and Chroma are vector stores too.",
    "Transformers power modern LLMs.",
]
len(texts)


### 3) Create embeddings
Use `text-embedding-3-small` for speed/cost.


In [ ]:
# OpenRouter embedding model
EMBED_MODEL = "openai/text-embedding-3-small"

def get_embedding(text: str):
    r = client.embeddings.create(model=EMBED_MODEL, input=text)
    return r.data[0].embedding

vectors = [get_embedding(t) for t in texts]
len(vectors), len(vectors[0])


### 4) Normalize and compare
We'll compute cosine similarities between pairs.


In [ ]:
import numpy as np

def l2_normalize(vec):
    v = np.array(vec, dtype=np.float32)
    n = np.linalg.norm(v) + 1e-10
    return v / n

normed = [l2_normalize(v) for v in vectors]

# cosine similarity matrix
S = np.matmul(np.stack(normed), np.stack(normed).T)
np.round(S, 3)


### 5) 2D visualization (optional)
Use PCA to plot rough clusters (tiny dataset, but illustrative).


In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

X = np.stack(normed)
pca = PCA(n_components=2)
xy = pca.fit_transform(X)

plt.figure(figsize=(4,3))
plt.scatter(xy[:,0], xy[:,1])
for i, label in enumerate(texts):
    plt.text(xy[i,0]+0.01, xy[i,1]+0.01, f"{i}")
plt.title("PCA of embeddings (indices shown)")
plt.show()


In [ ]:
texts